# Pipeline de Preço Justo — Materiais Recicláveis

**Objetivo:** Gerar a tabela de preço justo (R$/kg) por material × estado, a partir dos dados do BigQuery.

### Pipeline
1. Exportar dados do BigQuery (notas fiscais + catálogo de estoque)
2. Limpeza e seleção das colunas necessárias
3. Classificação de operações e TF-IDF matching contra catálogo
4. Clusterização por nome (TF-IDF hierárquico + desambiguação por preço)
5. Remoção de outliers e filtros de volume mínimo
6. Agregação mensal com lag-1m por material × estado
7. Geração da tabela de preço justo
8. Exportação dos resultados

> **Uso em produção:** Este notebook é projetado para execução periódica (cron a cada 2 semanas).
> Basta garantir que o  esteja no diretório correto.

## 1. Configuração e Dependências

In [1]:
!pip install google-cloud-bigquery db-dtypes

In [6]:
import re
import unicodedata
import warnings
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist
from scipy.sparse import hstack
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)

# ── Configuração ──
SERVICE_ACCOUNT_FILE = "../../service_account.json"  # ajustar caminho se necessário
PROJECT_ID = "analytics-big-query-242119"

# ── Constantes do pipeline ──
PRICE_COL = "unitpricekg_product"
VENDA_CATS = ["VENDA", "VENDA INTERESTADUAL"]
TFIDF_THRESHOLD = 0.25          # score mínimo para considerar um match TF-IDF
MATCH_SCORE_MIN = 0.6           # score mínimo para manter o registro
MIN_OBS_FAIR_PRICE = 5          # obs mínimas para entrar na tabela de preço justo
PRICE_RATIO_THRESHOLD = 2.0     # razão máx de preço antes de split no cluster
NAME_DIST_THRESHOLD = 0.50      # distância de corte para linkage hierárquico
MIN_MAT_OBS = 10                # obs mínimas por material
MIN_CLUSTER_OBS = 50            # obs mínimas por cluster

# ── Output ──
RUN_DATE = datetime.now().strftime("%Y-%m-%d")
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

print(f"✓ Configuração carregada — execução {RUN_DATE}")

✓ Configuração carregada — execução 2026-04-27


## 2. Exportação do BigQuery

In [ ]:
# ── Carregar dados de CSVs locais (modo teste) ──
# Em produção, substituir por queries ao BigQuery (ver comentário abaixo)

#COLS_INVOICES = [
#    "natop_operatorinvoice", "material", "description_product",
#    "emittedat_operatorinvoice", "unitpricekg_product",
#    "emitterstateuf", "emittercnae",
#]

#print("Carregando notas fiscais do CSV local...")
#df_invoices = pd.read_csv("products_invoices.csv", usecols=COLS_INVOICES)
#print(f"  → {len(df_invoices):,} linhas carregadas")

#print("Carregando catálogo de estoque do CSV local...")
#df_estoque = pd.read_csv("consulta_estoque.csv", usecols=["c_descricao"])
#df_estoque = df_estoque.dropna(subset=["c_descricao"]).drop_duplicates(subset=["c_descricao"])
#print(f"  → {len(df_estoque):,} produtos carregados")

# ── Versão BigQuery (produção) ──
creds = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE)
client = bigquery.Client(credentials=creds, project=PROJECT_ID)
QUERY_INVOICES = """
SELECT natop_operatorinvoice, material, description_product,
        emittedat_operatorinvoice, unitpricekg_product, emitterstateuf, emittercnae
 FROM `analytics-big-query-242119.dataform.products_invoices`
 WHERE unitpricekg_product IS NOT NULL
 """
QUERY_ESTOQUE = """
 SELECT DISTINCT c_descricao
 FROM `analytics-big-query-242119.omie_etl_hive.consulta_estoque`
 WHERE c_descricao IS NOT NULL
 """
df_invoices = client.query(QUERY_INVOICES).to_dataframe()
df_estoque = client.query(QUERY_ESTOQUE).to_dataframe()

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## 3. Limpeza e Preparação

In [10]:
# ── Parse de datas e filtro de anos ──
df_invoices["emittedat_operatorinvoice"] = pd.to_datetime(
    df_invoices["emittedat_operatorinvoice"], errors="coerce"
)
df_invoices["year"] = df_invoices["emittedat_operatorinvoice"].dt.year
df_invoices["month"] = df_invoices["emittedat_operatorinvoice"].dt.month
df_invoices["year_month"] = df_invoices["emittedat_operatorinvoice"].dt.to_period("M")
df_invoices = df_invoices[df_invoices["year"].isin([2023, 2024, 2025])].copy()

# ── Remover linhas sem dados essenciais ──
df_invoices[PRICE_COL] = pd.to_numeric(df_invoices[PRICE_COL], errors="coerce")
df_invoices = df_invoices.dropna(subset=[PRICE_COL, "description_product", "material"])
df_invoices = df_invoices[df_invoices[PRICE_COL] > 0]
df_invoices = df_invoices[df_invoices["material"].str.strip() != ""]
df_invoices = df_invoices[df_invoices["material"].str.strip().str.lower() != "unknown"]
df_invoices = df_invoices[df_invoices["description_product"].str.strip() != ""]

yr_min = df_invoices["year"].min()
yr_max = df_invoices["year"].max()
n_states = df_invoices["emitterstateuf"].nunique()
print(f"✓ Dados limpos: {len(df_invoices):,} notas fiscais válidas")
print(f"  Período: {yr_min}–{yr_max}")
print(f"  Estados: {n_states}")

✓ Dados limpos: 92,842 notas fiscais válidas
  Período: 2023–2025
  Estados: 27


/var/folders/mq/jmjwxty97tjbkgg_0tjt9fm80000gn/T/ipykernel_19184/2141253185.py:7: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_invoices["year_month"] = df_invoices["emittedat_operatorinvoice"].dt.to_period("M")


## 4. Classificação de Operações e TF-IDF Matching

In [18]:
# ── Funções auxiliares ──

def normalize_text(s):
    """Remove acentos, upper-case, normaliza espaços."""
    if pd.isna(s):
        return ""
    s = str(s).strip().upper()
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return " ".join(s.split())


def classify_natop(natop):
    """Classifica natureza da operação em VENDA/COMPRA/OUTROS."""
    if pd.isna(natop):
        return "OUTROS"
    s = str(natop).strip().lower()
    is_inter = any(kw in s for kw in [
        "interestadual", "inter estadual", "interstate",
        "fora do estado", "fora estado", "outros estados",
    ])
    if is_inter:
        if any(kw in s for kw in ["venda", "vda", "saida", "saída", "remessa", "sucata"]):
            return "VENDA INTERESTADUAL"
        if any(kw in s for kw in ["compra", "cpa", "entrada", "aquisicao", "aquisição"]):
            return "COMPRA INTERESTADUAL"
        return "OUTROS"
    if any(kw in s for kw in ["venda", "vda", "vnd"]):
        return "VENDA"
    if re.search(r"\bsaida\b|\bsaída\b", s) and not any(kw in s for kw in ["devolucao", "devolução", "retorno"]):
        return "VENDA"
    if any(kw in s for kw in ["compra", "cpa"]):
        return "COMPRA"
    if re.search(r"\bentrada\b", s) and not any(kw in s for kw in ["devolucao", "devolução", "retorno"]):
        return "COMPRA"
    if any(kw in s for kw in ["aquisicao", "aquisição"]):
        return "COMPRA"
    return "OUTROS"


def classify_material_group(text):
    """Classifica material em grupo de polímero."""
    if pd.isna(text):
        return "OUTROS"
    s = str(text).strip().upper()
    if "PEAD" in s:    return "PEAD"
    if "PET" in s:     return "PET"
    if "BOPP" in s or "RAFIA" in s: return "BOPP/RAFIA"
    if "PP" in s:      return "PP"
    if "PS" in s:      return "PS"
    if "PVC" in s:     return "PVC"
    if "PE " in s or "FILME" in s or s.startswith("PE"): return "PE/FILME"
    return "OUTROS"


_CNAE_RE = re.compile(r"^\d{2}\.\d{2}-\d-\d{2}$")

def classify_cnae_sector(cnae):
    """Classifica CNAE em setor simplificado."""
    if pd.isna(cnae):
        return "DESCONHECIDO"
    s = str(cnae).strip()
    if not _CNAE_RE.match(s):
        return "DESCONHECIDO"
    div = s[:2]
    if div == "38":                          return "COLETA_RECICLAGEM"
    if div in ("94", "88"):                  return "COOPERATIVA"
    if s.startswith("46.87") or s.startswith("46.86"): return "COMERCIO_ATACADO"
    if div in ("22", "17", "23", "24", "25", "20", "28", "13", "15", "26",
               "27", "29", "30", "31", "32", "33"): return "INDUSTRIA"
    if div.startswith("4"):                  return "COMERCIO"
    return "OUTROS_SETOR"


# ── Sinônimos para normalização TF-IDF ──
SYNONYMS = {
    "LATAS DE ALUMINIO": "LATINHAS", "LATA DE ALUMINIO": "LATINHAS",
    "LATINHAS DE ALUMINIO": "LATINHAS", "LATINHA DE ALUMINIO": "LATINHAS",
    "LATA ALUMINIO": "LATINHAS", "LATAS ALUMINIO": "LATINHAS",
    "ALUMINIO LATA": "LATINHAS", "LATINHA": "LATINHAS",
    "SACOLAS": "SACOLINHA", "SACOLA": "SACOLINHA", "SACOLINHAS": "SACOLINHA",
    "FILME PEBD": "FILME", "FILME PE": "FILME", "PEBD": "PE",
    "RAFFIA": "RAFIA", "BIG BAGS": "RAFIA", "BIG BAG": "RAFIA", "BIGBAG": "RAFIA",
    "COPOS": "COPINHO", "COPO": "COPINHO",
    "POTES MARGARINA": "MARGARINA", "POTE MARGARINA": "MARGARINA",
    "POTE DE MARGARINA": "MARGARINA",
    "BALDES": "BALDE BACIA", "BALDE": "BALDE BACIA",
    "BACIAS": "BALDE BACIA", "BACIA": "BALDE BACIA",
    "TAMPAS": "TAMPINHA", "TAMPA": "TAMPINHA", "TAMPINHAS": "TAMPINHA",
    "PRE-FORMA": "PREFORMA", "PRE FORMA": "PREFORMA",
    "BANDEJAS": "BANDEJA",
    "APARAS DE": "SUCATA DE", "APARA DE": "SUCATA DE",
    "APARAS": "SUCATA", "APARA": "SUCATA",
    "POS CONSUMO": "", "POS-CONSUMO": "", "PRE CONSUMO": "", "PRE-CONSUMO": "",
    "RECICLADO": "", "RECICLADA": "", "TRITURADO": "", "TRITURADA": "",
    "MOIDO": "", "MOIDA": "",
}
_SORTED_SYNS = sorted(SYNONYMS.items(), key=lambda x: -len(x[0]))

def expand_synonyms(text):
    for old, new in _SORTED_SYNS:
        text = re.sub(r"\b" + re.escape(old) + r"\b", new, text)
    return " ".join(text.split())

print("✓ Classificadores definidos")

✓ Classificadores definidos


In [19]:
# ── Classificar operações ──
df_invoices["natop_category"] = df_invoices["natop_operatorinvoice"].apply(classify_natop)
df_clean = df_invoices[df_invoices["natop_category"] != "OUTROS"].copy()

# ── Preparar catálogo de estoque para TF-IDF ──
df_estoque["material_group"] = df_estoque["c_descricao"].apply(classify_material_group)

estoque_products = sorted(set(
    p for p in df_estoque["c_descricao"].apply(normalize_text).unique() if p
))
estoque_expanded = [expand_synonyms(p) for p in estoque_products]
estoque_to_group = {}
for _, row in df_estoque.iterrows():
    key = normalize_text(row["c_descricao"])
    if key:
        estoque_to_group[key] = row["material_group"]

# ── Descrições únicas das notas ──
unique_descs = sorted(set(
    d for d in df_clean["description_product"].apply(normalize_text).unique() if d
))
descs_expanded = [expand_synonyms(d) for d in unique_descs]

# ── TF-IDF matching (char n-grams + word n-grams) ──
all_texts = estoque_expanded + descs_expanded
n_est = len(estoque_expanded)

vec_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
tfidf_char = vec_char.fit_transform(all_texts)
sim_char = cosine_similarity(tfidf_char[n_est:], tfidf_char[:n_est])

vec_word = TfidfVectorizer(analyzer="word", ngram_range=(1, 2))
tfidf_word = vec_word.fit_transform(all_texts)
sim_word = cosine_similarity(tfidf_word[n_est:], tfidf_word[:n_est])

sim_matrix = 0.5 * sim_char + 0.5 * sim_word

# ── Mapear cada descrição ao melhor produto do estoque ──
match_map = {}
for i, desc in enumerate(unique_descs):
    best_idx = sim_matrix[i].argmax()
    best_score = sim_matrix[i, best_idx]
    if best_score >= TFIDF_THRESHOLD:
        match_map[desc] = (estoque_products[best_idx], round(best_score, 3))
    else:
        match_map[desc] = ("SEM CORRESPONDENCIA", round(best_score, 3))

# ── Aplicar matching ao dataset ──
df_clean["desc_normalized"] = df_clean["description_product"].apply(normalize_text)
df_clean["estoque_product"] = df_clean["desc_normalized"].map(
    lambda d: match_map.get(d, ("SEM CORRESPONDENCIA", 0))[0]
)
df_clean["match_score"] = df_clean["desc_normalized"].map(
    lambda d: match_map.get(d, ("SEM CORRESPONDENCIA", 0))[1]
)
df_clean["material_group"] = df_clean["desc_normalized"].map(
    lambda d: estoque_to_group.get(match_map.get(d, ("", 0))[0], None)
)
# Fallback: classificar pelo campo material se não achou no estoque
mask_no_group = df_clean["material_group"].isna()
df_clean.loc[mask_no_group, "material_group"] = (
    df_clean.loc[mask_no_group, "material"].apply(classify_material_group)
)

# ── Filtrar por score e preço válido ──
df_price_clean = df_clean[df_clean["match_score"] > MATCH_SCORE_MIN].copy()

# ── Setor CNAE ──
df_price_clean["emitter_sector"] = df_price_clean["emittercnae"].apply(classify_cnae_sector)

print(f"✓ TF-IDF matching completo")
print(f"  Descrições únicas: {len(unique_descs):,}")
print(f"  Produtos no catálogo: {len(estoque_products):,}")
print(f"  Registros com match > {MATCH_SCORE_MIN}: {len(df_price_clean):,}")

✓ TF-IDF matching completo
  Descrições únicas: 5,762
  Produtos no catálogo: 93
  Registros com match > 0.6: 31,095


## 5. Clusterização por Nome (TF-IDF Hierárquico + Preço)

In [20]:
def cluster_by_name(df, group_col="material_group"):
    """
    Agrupa produtos por similaridade textual dentro de cada material_group.
    - TF-IDF hierárquico (char + word n-grams) → linkage average
    - Desambiguação por preço: split via KMeans se razão > PRICE_RATIO_THRESHOLD
    """
    df = df.copy()
    df["material_cluster"] = df[group_col]

    for group_name, group_df in df.groupby(group_col):
        prod_stats = (
            group_df.groupby("estoque_product")
            .agg(median_price=(PRICE_COL, "median"), count=(PRICE_COL, "count"))
            .reset_index()
        )
        if len(prod_stats) < 2:
            product = prod_stats["estoque_product"].iloc[0]
            df.loc[df[group_col] == group_name, "material_cluster"] = f"{group_name}::{product}"
            continue

        names = prod_stats["estoque_product"].apply(normalize_text).values
        vec_c = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
        vec_w = TfidfVectorizer(analyzer="word", ngram_range=(1, 2))
        X = hstack([vec_c.fit_transform(names), vec_w.fit_transform(names)])

        dist = np.nan_to_num(pdist(X.toarray(), metric="cosine"), nan=0.0)
        Z = linkage(dist, method="average")
        labels = fcluster(Z, t=NAME_DIST_THRESHOLD, criterion="distance")
        prod_stats["name_cluster"] = labels

        # Post-hoc: split clusters com preços muito diferentes
        final_labels = prod_stats["name_cluster"].copy()
        next_label = labels.max() + 1
        for cl in prod_stats["name_cluster"].unique():
            cl_prods = prod_stats[prod_stats["name_cluster"] == cl]
            if len(cl_prods) < 2:
                continue
            if cl_prods["median_price"].min() > 0 and (
                cl_prods["median_price"].max() / cl_prods["median_price"].min() > PRICE_RATIO_THRESHOLD
            ):
                log_p = np.log1p(cl_prods["median_price"].values).reshape(-1, 1)
                sub_labels = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(log_p)
                for sl in range(1, 2):
                    final_labels.loc[cl_prods.index[sub_labels == sl]] = next_label
                    next_label += 1

        prod_stats["final_cluster"] = final_labels

        # Nomear clusters pelo produto mais frequente
        cluster_names = {}
        for cl_id in prod_stats["final_cluster"].unique():
            top = prod_stats[prod_stats["final_cluster"] == cl_id].sort_values("count", ascending=False).iloc[0]
            top_name = top["estoque_product"]
            cluster_names[cl_id] = f"{group_name}::{top_name}"

        product_to_cluster = dict(
            zip(prod_stats["estoque_product"], prod_stats["final_cluster"].map(cluster_names))
        )
        mask = df[group_col] == group_name
        df.loc[mask, "material_cluster"] = df.loc[mask, "estoque_product"].map(product_to_cluster)
        df.loc[mask & df["material_cluster"].isna(), "material_cluster"] = group_name

    return df


df_price_clean = cluster_by_name(df_price_clean)

n_groups = df_price_clean["material_group"].nunique()
n_clusters = df_price_clean["material_cluster"].nunique()
print("✓ Clusterização completa")
print(f"  Grupos originais: {n_groups}")
print(f"  Clusters finais:  {n_clusters}")
for g in sorted(df_price_clean["material_group"].unique()):
    n = df_price_clean[df_price_clean["material_group"] == g]["material_cluster"].nunique()
    print(f"    {g}: {n} clusters")

✓ Clusterização completa
  Grupos originais: 8
  Clusters finais:  60
    BOPP/RAFIA: 5 clusters
    OUTROS: 12 clusters
    PE/FILME: 7 clusters
    PEAD: 8 clusters
    PET: 13 clusters
    PP: 11 clusters
    PS: 2 clusters
    PVC: 2 clusters


## 6. Remoção de Outliers e Filtros de Volume

In [21]:
def remove_outliers_iqr(df, col, groupby_cols, k=1.5):
    """Remove outliers por IQR dentro de cada grupo."""
    parts = []
    for _, grp in df.groupby(groupby_cols):
        q1, q3 = grp[col].quantile(0.25), grp[col].quantile(0.75)
        iqr = q3 - q1
        parts.append(grp[(grp[col] >= q1 - k * iqr) & (grp[col] <= q3 + k * iqr)])
    return pd.concat(parts, ignore_index=True)


# ── Filtrar apenas vendas ──
df_price_clean = df_price_clean[df_price_clean["natop_category"].isin(VENDA_CATS)].copy()

# ── Outlier removal por cluster × estado ──
n0 = len(df_price_clean)
df_price_clean = remove_outliers_iqr(df_price_clean, PRICE_COL, ["material_cluster", "emitterstateuf"])
print(f"✓ Outlier removal (IQR k=1.5): {n0:,} → {len(df_price_clean):,} ({n0 - len(df_price_clean):,} removidos)")

# ── Remover materiais raros ──
mat_counts = df_price_clean["estoque_product"].value_counts()
rare = mat_counts[mat_counts < MIN_MAT_OBS].index
n1 = len(df_price_clean)
df_price_clean = df_price_clean[~df_price_clean["estoque_product"].isin(rare)].copy()
print(f"✓ Materiais raros (< {MIN_MAT_OBS} obs): {n1:,} → {len(df_price_clean):,} ({len(rare)} materiais removidos)")

# ── Remover clusters pequenos ──
cl_counts = df_price_clean["material_cluster"].value_counts()
small = cl_counts[cl_counts < MIN_CLUSTER_OBS].index
n2 = len(df_price_clean)
df_price_clean = df_price_clean[~df_price_clean["material_cluster"].isin(small)].copy()
print(f"✓ Clusters pequenos (< {MIN_CLUSTER_OBS} obs): {n2:,} → {len(df_price_clean):,} ({len(small)} clusters removidos)")

# ── Remover pares cluster×estado com dados insuficientes ──
def quality_tier(row):
    if row["n_months"] >= 24 and row["n_rows"] >= 100: return "A (excellent)"
    if row["n_months"] >= 12 and row["n_rows"] >= 30:  return "B (good)"
    if row["n_months"] >= 6  and row["n_rows"] >= 10:  return "C (fair)"
    return "D (insufficient)"

coverage = (
    df_price_clean.groupby(["material_cluster", "emitterstateuf"])
    .agg(n_rows=(PRICE_COL, "count"), n_months=("year_month", "nunique"))
    .reset_index()
)
coverage["quality"] = coverage.apply(quality_tier, axis=1)
keep_pairs = coverage[coverage["quality"] != "D (insufficient)"][["material_cluster", "emitterstateuf"]]

n3 = len(df_price_clean)
df_price_clean = df_price_clean.merge(keep_pairs, on=["material_cluster", "emitterstateuf"], how="inner")
n_removed = len(coverage[coverage["quality"] == "D (insufficient)"])
print(f"✓ Pares D (insufficient) removidos: {n_removed} pares — {n3:,} → {len(df_price_clean):,} linhas")

n_cl = df_price_clean["material_cluster"].nunique()
n_st = df_price_clean["emitterstateuf"].nunique()
print(f"\n  Dataset final: {len(df_price_clean):,} registros, {n_cl} clusters, {n_st} estados")

✓ Outlier removal (IQR k=1.5): 30,378 → 28,986 (1,392 removidos)
✓ Materiais raros (< 10 obs): 28,986 → 28,927 (16 materiais removidos)
✓ Clusters pequenos (< 50 obs): 28,927 → 28,521 (16 clusters removidos)
✓ Pares D (insufficient) removidos: 149 pares — 28,521 → 27,742 linhas

  Dataset final: 27,742 registros, 33 clusters, 22 estados


## 7. Agregação Mensal + Lag-1M

In [22]:
# ── Agregar por mês × cluster × estado ──
monthly = (
    df_price_clean.groupby(["material_cluster", "emitterstateuf", "year", "month"])
    .agg(
        price_median=(PRICE_COL, "median"),
        price_mean=(PRICE_COL, "mean"),
        price_q25=(PRICE_COL, lambda x: x.quantile(0.25)),
        price_q75=(PRICE_COL, lambda x: x.quantile(0.75)),
        n_transactions=(PRICE_COL, "count"),
    )
    .reset_index()
)

# Price fair: midpoint IQR se distribuição simétrica, senão mediana
monthly["price_fair"] = monthly.apply(
    lambda r: r["price_median"]
    if r["price_median"] == 0 or abs(r["price_mean"] - r["price_median"]) / max(r["price_median"], 0.01) > 0.2
    else (r["price_q25"] + r["price_q75"]) / 2,
    axis=1,
)

monthly["date"] = pd.to_datetime(
    monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2) + "-01"
)

# ── Calcular lag-1m (preço do mês anterior) ──
lag_dfs = []
for (cluster, state), sub in monthly.groupby(["material_cluster", "emitterstateuf"]):
    sub = sub.sort_values("date").copy()
    sub["lag_1m"] = sub["price_median"].shift(1)
    lag_dfs.append(sub)
monthly = pd.concat(lag_dfs, ignore_index=True)

date_min = monthly["date"].min().strftime("%Y-%m")
date_max = monthly["date"].max().strftime("%Y-%m")
n_pairs = monthly.groupby(["material_cluster", "emitterstateuf"]).ngroups
print(f"✓ Agregação mensal: {len(monthly):,} observações mensais")
print(f"  Período: {date_min} a {date_max}")
print(f"  Clusters × Estados: {n_pairs}")

✓ Agregação mensal: 4,222 observações mensais
  Período: 2023-01 a 2025-12
  Clusters × Estados: 204


## 8. Tabela de Preço Justo

In [23]:
# ── Gerar tabela de preço justo ──
fair_prices = []

for (cluster, state), sub in df_price_clean.groupby(["material_cluster", "emitterstateuf"]):
    if len(sub) < MIN_OBS_FAIR_PRICE:
        continue

    q1, median, q3 = sub[PRICE_COL].quantile([0.25, 0.5, 0.75])
    cv = sub[PRICE_COL].std() / sub[PRICE_COL].mean() if sub[PRICE_COL].mean() > 0 else 0

    # Setor dominante
    sector_vc = sub["emitter_sector"].value_counts()
    dom_sector = sector_vc.index[0] if len(sector_vc) > 0 else "DESCONHECIDO"

    # Lag-1m do mês mais recente (previsão para o próximo mês)
    monthly_sub = monthly[
        (monthly["material_cluster"] == cluster) & (monthly["emitterstateuf"] == state)
    ].sort_values("date")
    last_month_price = monthly_sub["price_median"].iloc[-1] if len(monthly_sub) > 0 else None

    fair_prices.append({
        "cluster": cluster,
        "material_base": sub["material_group"].iloc[0],
        "state": state,
        "dominant_sector": dom_sector,
        "n_obs": len(sub),
        "n_months_data": sub["year_month"].nunique(),
        "n_products": sub["estoque_product"].nunique(),
        "median_price": round(median, 3),
        "Q1": round(q1, 3),
        "Q3": round(q3, 3),
        "fair_buy": round(q1, 3),
        "fair_sell": round(q3, 3),
        "fair_target": round(median, 3),
        "predicted_next": round(last_month_price, 3) if last_month_price else None,
        "heterogeneity": round(cv, 3),
        "top_products": ", ".join(sub["estoque_product"].value_counts().head(3).index.tolist()),
    })

df_fair = pd.DataFrame(fair_prices)

# ── Quality tier baseado em volume e heterogeneidade ──
def quality_tier_advanced(row):
    if row["n_obs"] >= 100 and row["heterogeneity"] < 0.3: return "A (excellent)"
    if row["n_obs"] >= 50  and row["heterogeneity"] < 0.5: return "B (good)"
    if row["n_obs"] >= 20:                                   return "C (fair)"
    return "D (insufficient)"

df_fair["quality"] = df_fair.apply(quality_tier_advanced, axis=1)

print("Distribuição de qualidade (antes do filtro):")
print(df_fair["quality"].value_counts().sort_index().to_string())
n_d = len(df_fair[df_fair["quality"] == "D (insufficient)"])
print(f"\n⚠ Removendo {n_d} pares D (insufficient)")

df_fair = df_fair[df_fair["quality"] != "D (insufficient)"].copy()
df_fair["run_date"] = RUN_DATE
df_fair = df_fair.sort_values(["material_base", "cluster", "state"]).reset_index(drop=True)

print(f"\n✓ Tabela de preço justo final: {len(df_fair)} pares (cluster × estado)")
print("\nDistribuição de qualidade (final):")
print(df_fair["quality"].value_counts().sort_index().to_string())
print("\nPreview:")
display(df_fair.head(20))

Distribuição de qualidade (antes do filtro):
quality
A (excellent)       37
B (good)            44
C (fair)            92
D (insufficient)    31

⚠ Removendo 31 pares D (insufficient)

✓ Tabela de preço justo final: 173 pares (cluster × estado)

Distribuição de qualidade (final):
quality
A (excellent)    37
B (good)         44
C (fair)         92

Preview:


,cluster,material_base,state,dominant_sector,n_obs,n_months_data,n_products,median_price,Q1,Q3,fair_buy,fair_sell,fair_target,predicted_next,heterogeneity,top_products,quality,run_date
0,BOPP/RAFIA::PP RAFIA,BOPP/RAFIA,PR,COOPERATIVA,127,35,1,0.30,0.200,0.400,0.200,0.400,0.30,0.400,0.335,PP RAFIA,B (good),2026-04-27
1,BOPP/RAFIA::SUCATA DE RAFIA,BOPP/RAFIA,PR,COOPERATIVA,56,27,1,0.30,0.200,0.362,0.200,0.362,0.30,0.350,0.332,SUCATA DE RAFIA,B (good),2026-04-27
2,BOPP/RAFIA::SUCATA DE RAFIA,BOPP/RAFIA,SC,COOPERATIVA,136,32,1,0.65,0.500,0.650,0.500,0.650,0.65,0.650,0.130,SUCATA DE RAFIA,A (excellent),2026-04-27
3,BOPP/RAFIA::SUCATA DE RAFIA,BOPP/RAFIA,SP,COMERCIO_ATACADO,65,25,1,0.30,0.100,0.300,0.100,0.300,0.30,0.225,0.479,SUCATA DE RAFIA,B (good),2026-04-27
4,OUTROS::BOMBONA,OUTROS,PR,COOPERATIVA,32,16,1,1.60,1.600,2.300,1.600,2.300,1.60,2.300,0.191,BOMBONA,C (fair),2026-04-27
5,OUTROS::LATA,OUTROS,MG,COOPERATIVA,21,14,1,7.00,6.000,9.000,6.000,9.000,7.00,10.000,0.212,LATA,C (fair),2026-04-27
6,OUTROS::LATA,OUTROS,SP,COMERCIO_ATACADO,25,15,1,10.20,9.200,10.800,9.200,10.800,10.20,10.800,0.138,LATA,C (fair),2026-04-27
7,OUTROS::PAPEL,OUTROS,BA,COOPERATIVA,88,30,1,0.35,0.292,0.500,0.292,0.500,0.35,0.500,0.366,PAPEL,B (good),2026-04-27
8,OUTROS::PAPEL,OUTROS,CE,COOPERATIVA,30,21,1,0.25,0.200,0.300,0.200,0.300,0.25,0.250,0.404,PAPEL,C (fair),2026-04-27
9,OUTROS::PAPEL,OUTROS,MG,COOPERATIVA,173,20,1,0.50,0.300,0.800,0.300,0.800,0.50,0.450,0.540,PAPEL,C (fair),2026-04-27


## 9. Exportação dos Resultados

In [24]:
# ── Salvar tabela de preço justo ──
output_dated = OUT_DIR / f"preco_justo_{RUN_DATE}.csv"
output_latest = OUT_DIR / "preco_justo_latest.csv"

df_fair.to_csv(output_dated, index=False)
df_fair.to_csv(output_latest, index=False)

# ── Resumo final ──
n_cl = df_price_clean["material_cluster"].nunique()
n_st = df_price_clean["emitterstateuf"].nunique()
print("=" * 60)
print("  PIPELINE CONCLUÍDO")
print("=" * 60)
print(f"\n  Data da execução:     {RUN_DATE}")
print(f"  Registros de entrada:  {len(df_invoices):,} notas fiscais")
print(f"  Registros processados: {len(df_price_clean):,} (pós filtros)")
print(f"  Clusters:              {n_cl}")
print(f"  Estados:               {n_st}")
print(f"  Pares na tabela:       {len(df_fair)}")
print("\n  Qualidade:")
for q in sorted(df_fair["quality"].unique()):
    n = len(df_fair[df_fair["quality"] == q])
    print(f"    {q}: {n} pares")
print(f"\n  Arquivos salvos:")
print(f"    {output_dated}")
print(f"    {output_latest}")
print("\n✓ Pronto para consumo downstream.")

  PIPELINE CONCLUÍDO

  Data da execução:     2026-04-27
  Registros de entrada:  92,842 notas fiscais
  Registros processados: 27,742 (pós filtros)
  Clusters:              33
  Estados:               22
  Pares na tabela:       173

  Qualidade:
    A (excellent): 37 pares
    B (good): 44 pares
    C (fair): 92 pares

  Arquivos salvos:
    output/preco_justo_2026-04-27.csv
    output/preco_justo_latest.csv

✓ Pronto para consumo downstream.
